In [ ]:
# configuration guide
# This is a "living template" to remember what you can tweak in env, models, and training.

ENV_OPTIONS = {
    "observation_profile": "minimal_basic_strategy",  # "minimal_basic_strategy" | "table_realistic_default" | "table_realistic_unknown_progress" | "fully_observable_sim"
    "start_state_mode": "fresh_shoe",                 # "fresh_shoe" | "unknown_progress"
    "n_decks": 1,                                     # int > 0, number of decks
    "shoe_penetration": 1.0,                          # float in (0, 1], how deep the shoe goes before reshuffling
    "dealer_hits_soft_17": False,                     # whether dealer hits on soft 17
    "blackjack_payout": 1.5,                          # blackjack payout
    "dealer_peeks_for_blackjack": True,               # whether dealer peeks for blackjack
    "double_allowed_on": "any_two_cards",             # "any_two_cards" | "hard_9_10_11" | "hard_10_11"
    "double_after_split_allowed": True,               # allow double after split
    "split_rule": "same_value",                       # "same_rank" | "same_value"
    "max_hands_after_split": 4,                       # maximum hands after split
    "resplit_aces_allowed": True,                     # allow resplitting aces
    "hit_split_aces_allowed": False,                  # allow hitting after splitting aces
    "surrender_allowed": True,                        # allow surrender
    "insurance_allowed": True,                        # allow insurance
    "base_bet": 1.0,                                  # base bet
    "seed": 11,                                       # table seed
    "shoe": None,                                     # list of cards to fix the shoe, or None to leave it random
}

START_STATE_OPTIONS = {
    "mode": "unknown_progress",                       # "fresh_shoe" | "unknown_progress"
    "min_burned_rounds": 2,                           # minimum burned rounds before starting
    "max_burned_rounds": 2,                           # maximum burned rounds before starting
    "clear_visible_histories_after_burn": True,       # clear visible history after burning rounds
    "hide_reshuffle_progress_from_observation": True, # hide shoe progress from observation
}

MODEL_OPTIONS = {
    "architecture": "feedforward",                    # "feedforward" | "recurrent" | "dueling_recurrent"
    "encoder_profile": "minimal_basic_strategy",      # must be consistent with the observation
    "activation": "relu",                             # "relu" | "gelu"
    "use_layer_norm": False,                          # especially useful in recurrent networks
    "dropout": 0.0,                                   # usually 0 in DQN at the start
    "feedforward_hidden_dims": (256, 256),            # feedforward only
    "projection_dim": 256,                            # recurrent only
    "recurrent_hidden_dim": 256,                      # recurrent only
    "recurrent_num_layers": 1,                        # recurrent only
    "recurrent_type": "gru",                          # "gru" | "lstm"
    "head_hidden_dim": 128,                           # simple recurrent
    "value_hidden_dim": 128,                          # dueling recurrent
    "advantage_hidden_dim": 128,                      # dueling recurrent
}

TRAINING_OPTIONS = {
    "total_epochs": 2,                                # how many epochs to run
    "env_steps_per_epoch": 12,                        # environment steps per epoch
    "train_frequency": 1,                             # train frequency in env steps
    "updates_per_train_step": 1,                      # how many backward passes per train event
    "max_updates_per_epoch": 1,                       # limit to avoid full training
    "device": "cpu",                                  # "auto" | "cpu" | "cuda"
    "seed": 13,                                       # trainer seed
    "reset_hidden_on_round_end": False,               # recurrent: reset hidden state on round end
    "sequence_end_on_done": False,                    # recurrent: cut sequence when done=True
    "flush_partial_sequences_at_epoch_end": True,     # recurrent: flush partial sequences at epoch end

    "buffer_capacity": 64,                            # total capacity of the replay buffer
    "batch_size": 4,                                  # typical feedforward: >= 4; recurrent can be 1 for quick testing
    "warmup_size": 8,                                 # how much experience to collect before training
    "sequence_length": 4,                             # recurrent: maximum segment length
    "min_sequence_length": 2,                         # recurrent: minimum length to save segment

    "epsilon_start": 1.0,                             # initial epsilon
    "epsilon_end": 0.2,                               # final epsilon
    "epsilon_decay_steps": 20,                        # steps for decay
    "evaluation_epsilon": 0.0,                        # epsilon during evaluation

    "optimizer": "adam",                              # "adam" | "adamw"
    "learning_rate": 1e-3,                            # learning rate
    "weight_decay": 0.0,                              # L2 regularization
    "scheduler": "none",                              # "none" | "step"
    "scheduler_step_size": 1000,                      # if using StepLR
    "scheduler_gamma": 0.99,                          # scheduler decay
    "gradient_clipping": True,                        # gradient clipping
    "max_grad_norm": 5.0,                             # maximum norm if clipping

    "target_update_mode": "hard",                     # "hard" | "soft"
    "hard_update_interval": 2,                        # synchronization interval for target updates
    "soft_tau": 0.005,                                # tau if using soft update

    "loss_gamma": 0.99,                               # Bellman gamma
    "loss_type": "huber",                             # "huber" | "mse"

    "evaluation_enabled": False,                      # turn off for quick tests
    "evaluation_every_n_epochs": 1,                   # evaluation frequency
    "evaluation_num_rounds": 4,                       # eval rounds
    "evaluation_max_decisions": 200,                  # maximum decisions in eval

    "save_latest": False,                             # save latest checkpoint
    "save_best_eval": False,                          # save best checkpoint
    "save_periodic": False,                           # save periodic checkpoints
    "periodic_interval_updates": 1000,                # interval if save_periodic=True

    "prints_enable": False,                           # print logs during training
    "print_update_interval": 50,                      # update print interval
    "print_collection_interval": 100,                 # collection print interval
}


In [ ]:
from enviroment_bj import BlackjackConfig, ObservationConfig, StartStateConfig
from model.agents import FeedForwardDoubleDQN, RecurrentDoubleDQN, DuelingRecurrentDoubleDQN
from training import (
    ReplayBufferConfig,
    EpsilonScheduleConfig,
    OptimizationConfig,
    TargetUpdateConfig,
    EvaluationConfig,
    CheckpointConfig,
    PrintConfig,
    TrainerConfig,
    TrainingPipelineConfig,
)

# =========================
# 1) OBSERVATION / TABLE
# =========================

observation_config = ObservationConfig.for_profile(
    "table_realistic_default"  # "minimal_basic_strategy" | "table_realistic_default" | "table_realistic_unknown_progress" | "fully_observable_sim"
)

start_state_config = StartStateConfig(
    mode="fresh_shoe",                      # "fresh_shoe" | "unknown_progress"
    min_burned_rounds=0,                    # int >= 0, minimum rounds to burn before starting
    max_burned_rounds=0,                    # int >= min_burned_rounds, maximum rounds to burn
    clear_visible_histories_after_burn=True,    # True/False, clear visible history after burning rounds
    hide_reshuffle_progress_from_observation=False,  # True/False, hide shoe progress from observation
)

blackjack_config = BlackjackConfig(
    n_decks=1,                              # int > 0, number of decks
    shoe_penetration=1.0,                   # float in (0, 1], shoe penetration depth before reshuffle
    dealer_hits_soft_17=False,              # True/False
    blackjack_payout=1.5,                   # blackjack payout
    dealer_peeks_for_blackjack=True,        # True/False
    double_allowed_on="any_two_cards",      # "any_two_cards" | "hard_9_10_11" | "hard_10_11"
    double_after_split_allowed=True,        # True/False
    split_rule="same_value",                # "same_rank" | "same_value"
    max_hands_after_split=4,                # int >= 2
    resplit_aces_allowed=True,              # True/False
    hit_split_aces_allowed=False,           # True/False
    surrender_allowed=True,                 # True/False
    insurance_allowed=True,                 # True/False
    base_bet=1.0,                           # float > 0
    strict_shoe_validation=False,           # True/False, validate shoe composition
    observation=observation_config,         # ObservationConfig(...)
    observation_mode=None,                  # None | "basic_strategy" | "table_raw"
    expose_shoe_composition=False,          # True/False, force exact shoe composition
)

# =========================
# 2) MODEL
# =========================

feedforward_model = FeedForwardDoubleDQN.from_profile(
    "minimal_basic_strategy",               # encoder profile
    activation="relu",                      # "relu" | "gelu"
    use_layer_norm=False,                   # True/False
    dropout=0.0,                            # float in [0, 1)
    feedforward_hidden_dims=(256, 256),     # tuple[int, ...]
)

recurrent_model = RecurrentDoubleDQN.from_profile(
    "table_realistic_default",              # encoder profile
    activation="relu",                      # "relu" | "gelu"
    use_layer_norm=True,                    # True/False
    dropout=0.0,                            # float in [0, 1)
    projection_dim=256,                     # int > 0
    recurrent_hidden_dim=256,               # int > 0
    recurrent_num_layers=1,                 # int > 0
    recurrent_type="gru",                   # "gru" | "lstm"
    head_hidden_dim=128,                    # int > 0
)

dueling_recurrent_model = DuelingRecurrentDoubleDQN.from_profile(
    "table_realistic_unknown_progress",     # encoder profile
    activation="relu",                      # "relu" | "gelu"
    use_layer_norm=True,                    # True/False
    dropout=0.0,                            # float in [0, 1)
    projection_dim=256,                     # int > 0
    recurrent_hidden_dim=256,               # int > 0
    recurrent_num_layers=1,                 # int > 0
    recurrent_type="lstm",                  # "gru" | "lstm"
    value_hidden_dim=128,                   # int > 0
    advantage_hidden_dim=128,               # int > 0
)

# =========================
# 3) TRAINING PIPELINE
# =========================

replay_buffer_config = ReplayBufferConfig(
    capacity=64,                            # total size of the replay buffer
    batch_size=4,                           # batch size when sampling
    warmup_size=8,                          # minimum experience before training
    sequence_length=4,                      # recurrent: maximum segment length
    min_sequence_length=2,                  # recurrent: minimum length to save segment
)

epsilon_config = EpsilonScheduleConfig(
    start=1.0,                              # initial epsilon
    end=0.2,                                # final epsilon
    decay_steps=20,                         # steps to decay from start to end
    evaluation_epsilon=0.0,                 # epsilon used in evaluation
)

optimization_config = OptimizationConfig(
    optimizer="adam",                       # "adam" | "adamw"
    learning_rate=1e-3,                     # lr
    weight_decay=0.0,                       # L2 regularization
    scheduler="none",                       # "none" | "step"
    scheduler_step_size=1000,               # if scheduler="step"
    scheduler_gamma=0.99,                   # if scheduler="step"
    gradient_clipping=True,                 # True/False
    max_grad_norm=5.0,                      # maximum norm if clipping=True
)

target_update_config = TargetUpdateConfig(
    mode="hard",                            # "hard" | "soft"
    hard_update_interval=2,                 # synchronization interval if mode="hard"
    soft_tau=0.005,                         # tau if mode="soft"
)

evaluation_config = EvaluationConfig(
    enabled=False,                          # True/False
    every_n_epochs=1,                       # evaluate every N epochs
    num_rounds=4,                           # evaluation rounds
    max_decisions=200,                      # limit of decisions in evaluation
)

checkpoint_config = CheckpointConfig(
    directory="tmp_unused",                 # checkpoint directory
    save_latest=False,                      # True/False
    save_best_eval=False,                   # True/False
    save_periodic=False,                    # True/False
    periodic_interval_updates=1000,         # if save_periodic=True
    best_metric_name="ev_per_1000_hands",   # metric for the best model
    maximize_best_metric=True,              # True if the metric is maximized
)

print_config = PrintConfig(
    enable=False,                           # True/False
    print_update_interval=50,               # print every N updates
    print_collection_interval=100,          # print every N collections
    print_epoch_summary=True,               # True/False
    print_eval_summary=True,                # True/False
    include_segment_details=False,          # True/False, more detail for recurrent models
)

trainer_config = TrainerConfig(
    total_epochs=2,                         # total epochs
    env_steps_per_epoch=12,                 # environment steps per epoch
    train_frequency=1,                      # train every N env steps
    updates_per_train_step=1,               # how many backward passes per train event
    max_updates_per_epoch=1,                # limit updates per epoch for quick testing
    device="cpu",                           # "auto" | "cpu" | "cuda"
    seed=13,                                # global trainer seed
    reset_hidden_on_round_end=False,        # recurrent: reset hidden state on round end
    sequence_end_on_done=False,             # recurrent: cut sequence when done=True
    flush_partial_sequences_at_epoch_end=True,  # recurrent: flush partial sequences at epoch end
)

pipeline_config = TrainingPipelineConfig(
    trainer=trainer_config,
    replay_buffer=replay_buffer_config,
    epsilon=epsilon_config,
    optimization=optimization_config,
    target_update=target_update_config,
    evaluation=evaluation_config,
    checkpoints=checkpoint_config,
    prints=print_config,
)